# 📦 Notebook 01 — Data Loading & Schema Validation
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the PubMedQA dataset from HuggingFace
- Validate schema: confirm `question`, `context` (list), `long_answer`, `final_decision` columns exist
- Log raw row count, null counts, and a 5-row sample
- Save raw data locally as `data/raw/pubmedqa_raw.csv`
- Generate `reports/schema_validation_report.md`

### 📋 Deliverables
- `data/raw/pubmedqa_raw.csv`
- `reports/schema_validation_report.md`

---

## 1. Imports & Configuration

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import load_dataset
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# tabulate is required by DataFrame.to_markdown() used in the report cell.
# If missing: pip install tabulate
try:
    import tabulate  # noqa: F401
except ImportError:
    raise ImportError("Run:  pip install tabulate  then restart the kernel.")

# ── Anchor paths to the project root (works regardless of CWD) ─────────────
NOTEBOOK_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORTS_DIR  = PROJECT_ROOT / 'reports'
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RAW_DATA_DIR / 'pubmedqa_raw.csv'
REPORT_PATH = REPORTS_DIR  / 'schema_validation_report.md'

# ── Dataset config ──────────────────────────────────────────────────────────
DATASET_NAME      = 'qiaojin/PubMedQA'
DATASET_SUBSET    = 'pqa_artificial'
EXPECTED_COLUMNS  = ['pubid', 'question', 'context', 'long_answer', 'final_decision']
EXPECTED_ROW_COUNT = 211_000

print('✅ Imports successful')
print(f'📁 Raw data → {OUTPUT_PATH}')
print(f'📄 Report  → {REPORT_PATH}')

## 2. Load Dataset from HuggingFace

In [ ]:
print(f'⏳ Loading dataset: {DATASET_NAME} ({DATASET_SUBSET}) ...')

dataset = load_dataset(DATASET_NAME, DATASET_SUBSET)

print('\n✅ Dataset loaded successfully!')
print('\n📊 Dataset overview:')
print(dataset)

## 3. Inspect Available Splits

In [ ]:
print('🔍 Available splits:', list(dataset.keys()))
print()

for split_name, split_data in dataset.items():
    print(f'  Split: "{split_name}" → {len(split_data):,} records')

## 4. Convert to Pandas DataFrame

In [ ]:
main_split = list(dataset.keys())[0]
print(f'📌 Using split: "{main_split}"')

df = pd.DataFrame(dataset[main_split])

print(f'\n✅ Converted to DataFrame')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 5. Schema Validation

In [ ]:
print('🔍 Schema Validation')
print('─' * 50)

# Check columns
actual_columns = df.columns.tolist()
missing_cols = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_cols   = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

print(f'Expected columns: {EXPECTED_COLUMNS}')
print(f'Actual columns:   {actual_columns}')
print(f'Missing columns:  {missing_cols if missing_cols else "None"}')
print(f'Extra columns:    {extra_cols if extra_cols else "None"}')

if not missing_cols and not extra_cols:
    schema_status = f'PASSED — all {len(EXPECTED_COLUMNS)} expected columns present, no drift'
    print(f'\n✅ {schema_status}')
else:
    schema_status = f'FAILED — Missing: {missing_cols}, Extra: {extra_cols}'
    print(f'\n❌ {schema_status}')

# Check row count
print(f'\nExpected ~{EXPECTED_ROW_COUNT:,} rows, got {len(df):,} rows')
if len(df) >= EXPECTED_ROW_COUNT:
    print('✅ Row count OK')
else:
    print(f'⚠️  Fewer rows than expected')

# Check data types
print(f'\n📋 Data Types:')
print(df.dtypes)

## 6. Null Counts

In [ ]:
print('🔎 Missing Values per Column:')
print('─' * 40)

missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %':     missing_pct
})

print(missing_df)
print()

if missing.sum() == 0:
    null_status = 'No missing values found'
    print(f'✅ {null_status}')
else:
    null_status = f'{missing.sum():,} total missing values'
    print(f'⚠️  {null_status}')

## 7. Duplicate Check

In [ ]:
# Convert to string to handle unhashable types (e.g. list-of-dict in context)
n_duplicates = df.astype(str).duplicated().sum()
print(f'🔎 Duplicate rows: {n_duplicates:,}')

if n_duplicates == 0:
    dup_status = 'No duplicates found'
    print(f'✅ {dup_status}')
else:
    dup_status = f'{n_duplicates} duplicate rows found'
    print(f'⚠️  {dup_status}')

## 8. 5-Row Sample

In [ ]:
print('👀 First 5 records:')
pd.set_option('display.max_colwidth', 150)
df.head(5)

## 9. Basic Text Statistics

In [ ]:
# 'context' is a list-of-dict in the raw HuggingFace dataset, so we
# use select_dtypes only on proper string columns.
text_cols = ['question', 'long_answer']

print('📏 Text Length Statistics (character count):')
print('─' * 50)

for col in text_cols:
    lengths = df[col].astype(str).str.len()
    print(f'\n[{col}]')
    print(f'  Min:    {lengths.min():,}')
    print(f'  Max:    {lengths.max():,}')
    print(f'  Mean:   {lengths.mean():,.0f}')
    print(f'  Median: {lengths.median():,.0f}')

## 10. Save Raw Data

> ⚠️ **Note on `context` column:** PubMedQA's `context` column is a nested
> dict/list structure. `to_csv()` serialises it as a Python repr string.
> `02_preprocessing.ipynb` handles deserialisation via
> `src.data.preprocessor.run_preprocessing_pipeline`.

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

file_size = os.path.getsize(OUTPUT_PATH) / 1024**2

print(f'✅ Raw dataset saved to: {OUTPUT_PATH}')
print(f'   Rows:      {df.shape[0]:,}')
print(f'   Columns:   {df.shape[1]}')
print(f'   File size: {file_size:.1f} MB')

In [ ]:
# ── Optional: sync raw data to HuggingFace Hub ─────────────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

if not os.getenv('HF_TOKEN'):
    print("⚠️  HF_TOKEN not set — skipping upload.")
    print("   Add it to your .env file (see .env.example).")
else:
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.data.hub import upload_all_data
    print("\n📤 Syncing to HuggingFace...")
    upload_all_data()

## 11. Generate Schema Validation Report

In [ ]:
sample_md = df.head(5).to_markdown(index=False)

report = f"""# Schema Validation Report
**Healthcare RAG-Powered Medical Q&A Assistant**

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Dataset Info
| Item | Value |
|---|---|
| Source | `{DATASET_NAME}` |
| Split | `{main_split}` |
| Row count | {len(df):,} |
| Expected rows | ~{EXPECTED_ROW_COUNT:,} |

## Schema Validation
| Item | Value |
|---|---|
| Expected columns | {EXPECTED_COLUMNS} |
| Actual columns | {actual_columns} |
| Missing columns | {missing_cols if missing_cols else 'None'} |
| Extra columns | {extra_cols if extra_cols else 'None'} |
| **Status** | **{schema_status}** |

## Null Counts
{missing_df.to_markdown()}

**Status:** {null_status}

## Duplicates
| Item | Value |
|---|---|
| Duplicate rows | {n_duplicates} |
| **Status** | **{dup_status}** |

## 5-Row Sample
{sample_md}
"""

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(f'✅ Schema validation report saved to: {REPORT_PATH}')

## ✅ Summary

In [ ]:
# FIX: was a Markdown cell — Python f-strings don't evaluate in Markdown.
# Converted to a code cell so variables resolve correctly.
from IPython.display import Markdown, display

summary = f"""
| Item | Result |
|---|---|
| Dataset | `qiaojin/PubMedQA` (`pqa_artificial` subset) |
| Records | {len(df):,} |
| Dataset Subset | `{DATASET_SUBSET}` |
| Columns | {actual_columns} |
| Schema | {schema_status} |
| Missing values | {null_status} |
| Duplicates | {dup_status} |
| Saved raw CSV | `data/raw/pubmedqa_raw.csv` |
| Saved report | `reports/schema_validation_report.md` |
"""

display(Markdown(summary))
print()
print("➡️  Next Step: Open 02_preprocessing.ipynb")